### Structured Output

Models can be requested to provide their response in a format matching a given scheme. This is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested stuructures.

In [10]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001EFA304ACF0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EFA304BA10>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [11]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This is the year of release of the movie")
    director : str = Field(description="This is the name of the director")
    rating : str = Field(description="The imdb movie rating out of 10")
    actor : str = Field(description="This gives name of the main actor and actress")

In [12]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001EFA304ACF0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EFA304BA10>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This is the year of release of the movie', 'type': 'integer'}, 'director': {'description': 'This is the name of the director', 'type': 'string'}, 'rating': {'description': 'The im

In [13]:
model.invoke("Provide details about the movie Inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. Inception is a 2010 science fiction action film directed by Christopher Nolan. The main cast includes Leonardo DiCaprio, Joseph Gordon-Levitt, Ellen Page, and Tom Hardy. The plot revolves around the concept of entering someone\'s dreams to plant an idea, which is called "inception." \n\nThe director is definitely Christopher Nolan, known for other films like The Dark Knight and Interstellar. The main character is Dom Cobb, played by DiCaprio, who is a professional thief who steals information by infiltrating the subconscious of his targets. Instead of stealing, he\'s now asked to perform the reverse: planting an idea. The target is Robert Fischer, a business heir. The team uses a machine called the PASIV device, which allows them to enter multiple layers of dreams. \n\nThere\'s a lot of action, especially in the scenes where they\'re in a dream. The 

In [14]:
response = model_with_structure.invoke("Provide details about the movie Inception")

In [15]:
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating='8.8', actor='Leonardo DiCaprio, Ellen Page')

#### Message Output along side parsed output

In [16]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="This is the year of release of the movie")
    director : str = Field(..., description="This is the name of the director")
    rating : str = Field(..., description="The imdb movie rating out of 10")
    actor : str = Field(..., description="This gives name of the main actor and actress")

In [17]:
model_with_structure = model.with_structured_output(Movie,include_raw=True)
model_with_structure

{
  raw: _ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001EFA304ACF0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EFA304BA10>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This is the year of release of the movie', 'type': 'integer'}, 'director': {'description': 'This is the name of the director', 'type': 'string'}, 'rating': {'description'

In [18]:
response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check what tools I have available. There\'s a Movie function that requires title, year, director, rating, and actor. I need to make sure I get all these details for Inception.\n\nFirst, the title is obviously "Inception". The year it was released was 2010. The director is Christopher Nolan. The main actors are Leonardo DiCaprio, Joseph Gordon-Levitt, and Ellen Page. The IMDb rating is around 8.8. Let me verify these facts to be accurate. \n\nWait, is the year correct? Yes, Inception came out in 2010. Christopher Nolan directed it. The actors are correct. The rating might be 8.8 as of recent data. Once I confirm these details, I can structure the function call with all required parameters. Need to format the actors as a string, maybe list them separated by commas. The rating is a string here, so "8.8". The director is a single string. All required

#### Nested Structure

In [19]:
from pydantic import BaseModel, Field  # 👈 Added Field to imports

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions of USD")

# 👇 Out-dented (placed outside the class definition)
model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Inception")
response


MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Balthazar')], genres=['Science Fiction', 'Action', 'Heist'], budget=160.0)

### Typedict 

Typedict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation

In [20]:
from typing_extensions import TypedDict, Annotated

class Movie(TypedDict):
    '''A movie with details'''
    title: Annotated[str,...,"The title of the movie"]
    year: Annotated[int,..., "This is the year of release of the movie"]
    director : Annotated[str,..., "This is the name of the director"]
    rating : Annotated[float,..., "The imdb movie rating out of 10"]
    actor : Annotated[str,..., "This gives name of the main actor and actress"]

In [21]:
model_with_typedict = model.with_structured_output(Movie)
model_with_typedict.invoke("Please provide the details of the movie The Iron Man")

{'actor': 'Jon Doe, Emily Smith',
 'director': 'James Cameron',
 'rating': 7.5,
 'title': 'The Iron Man',
 'year': 1991}

In [22]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions of USD")

# 👇 Out-dented (placed outside the class definition)
model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie The Incrdible Hulk")
response


{'budget': 150000000,
 'cast': [{'name': 'Edward Norton', 'role': 'Bruce Banner / The Hulk'},
  {'name': 'Eric Bana', 'role': 'David Banner (younger Bruce Banner)'},
  {'name': 'Liv Tyler', 'role': 'Betty Ross'},
  {'name': 'Nick Nolte', 'role': "General Thaddeus 'Thunderbolt' Ross"}],
 'genres': ['Action', 'Superhero', 'Science Fiction'],
 'title': 'The Incredible Hulk',
 'year': 2008}

In [23]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### DataClasses

A dataclasss is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

In [24]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field

class ContactInfo(BaseModel):
    """Contact Information of a person"""
    name: str = Field(description="The name of a person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

model = ChatGroq(model="qwen/qwen3-32b")

structured_model = model.with_structured_output(ContactInfo)

result = structured_model.invoke([
    HumanMessage(content="Extract contact info from: John Doe, john@example.com, (555) 123-4567")
]) 

print(result)         # ContactInfo object
print(result.name)    # John Doe
print(result.email)   # john@example.com
print(result.phone)   # (555) 123-4567

name='John Doe' email='john@example.com' phone='(555) 123-4567'
John Doe
john@example.com
(555) 123-4567
